# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [21]:
import os

# 1. استنساخ المستودع من GitHub
print("⏳ جاري استنساخ المستودع من GitHub...")
!git clone https://github.com/maram-elaian/brandora.git

# 2. الدخول إلى مجلد المشروع
os.chdir('/kaggle/working/brandora')

# 3. التأكد من وجود الملف
if os.path.exists('data/test_briefs.json'):
    print("✅ تم الدخول إلى مجلد المشروع بنجاح!")
    print(f"📍 المسار الحالي: {os.getcwd()}")
    print("📂 ملف test_briefs.json موجود وجاهز.")
else:
    print("❌ تحذير: لم يتم العثور على الملف. تأكد من رفعه على GitHub.")

⏳ جاري استنساخ المستودع من GitHub...
fatal: destination path 'brandora' already exists and is not an empty directory.
✅ تم الدخول إلى مجلد المشروع بنجاح!
📍 المسار الحالي: /kaggle/working/brandora
📂 ملف test_briefs.json موجود وجاهز.


In [24]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة مفتاح OpenRouter بأمان
user_secrets = UserSecretsClient()
or_key = user_secrets.get_secret("OPENROUTER_API_KEY")

# 2. تهيئة العميل
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=or_key,
)

# 3. قائمة النماذج المجانية العاملة حالياً (محدثة)
models_to_test = [
    "liquid/lfm-2.5-2.6b:free",
    "qwen/qwen3-4b:free"
]

print("✅ تم الاتصال بـ OpenRouter بنجاح!")
print(f"🤖 النماذج المجانية الجاهزة للاختبار: {len(models_to_test)}")
for i, model in enumerate(models_to_test, 1):
    print(f"   {i}. {model}")

✅ تم الاتصال بـ OpenRouter بنجاح!
🤖 النماذج المجانية الجاهزة للاختبار: 2
   1. liquid/lfm-2.5-2.6b:free
   2. qwen/qwen3-4b:free


In [25]:
# تحميل البيانات (تأكدي أن خلية git clone قد عملت بنجاح قبل هذا)
with open('data/test_briefs.json', 'r', encoding='utf-8') as file:
    test_briefs = json.load(file)

system_prompt = """You are an expert Brand Strategist. 
Your task is to analyze a raw brand brief and extract a structured Brand Specification.
You MUST output ONLY valid JSON. Do not include any markdown formatting or extra text."""

sample_brief = test_briefs[0]

user_prompt = f"""Analyze this brief:
Industry: {sample_brief['industry']}
Target Audience: {sample_brief['target_audience']}
Description: {sample_brief['brand_purpose']}

Return a JSON object with EXACTLY this structure:
{{
  "industry": "string",
  "target_audience": "string",
  "brand_purpose": "string",
  "personality": ["string", "string"],
  "tone": "string",
  "visual_style": ["string", "string"],
  "keywords": ["string", "string"],
  "color_direction": ["string", "string"],
  "logo_direction": "string"
}}"""

print(f"⏳ جاري اختبار النموذج: {models_to_test[0]} ...\n")

# إرسال الطلب عبر OpenRouter
response = client.chat.completions.create(
    model=models_to_test[0],
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2
)

# استخراج وتنظيف النتيجة
raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result = json.loads(clean_json)

print("✅ نجح النموذج! إليك النتيجة:")
print(json.dumps(result, indent=2, ensure_ascii=False))

⏳ جاري اختبار النموذج: liquid/lfm-2.5-2.6b:free ...

✅ نجح النموذج! إليك النتيجة:
{
  "industry": "coffee",
  "target_audience": "university students",
  "brand_purpose": "To create a comfortable, energetic space that inspires focused learning and meaningful social connections for university students.",
  "personality": [
    "friendly",
    "vibrant",
    "modern",
    "welcoming",
    "dynamic"
  ],
  "tone": "Approachable and invigorating",
  "visual_style": [
    "clean minimalism",
    "bold typography",
    "warm earthy tones",
    "energetic accent colors"
  ],
  "keywords": [
    "study-friendly",
    "social",
    "energetic",
    "comfortable",
    "modern",
    "academic",
    "community"
  ],
  "color_direction": [
    "warm neutrals",
    "electric blue accents",
    "sunny yellow highlights"
  ],
  "logo_direction": "An iconic coffee cup combined with abstract geometric shapes that symbolize connection and movement"
}


In [26]:
# Test Qwen3-4B

qwen_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=or_key,
)

response = qwen_client.chat.completions.create(
    model="qwen/qwen3-4b:free",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2
)

qwen_raw = response.choices[0].message.content

print("Qwen3-4B output:")
print(qwen_raw)

NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found for qwen/qwen3-4b:free.', 'code': 404}, 'user_id': 'user_3Jdw3so80KuHsM00WxgBhViWBKx'}

In [18]:
import json

# فتح ملف الاختبار
with open('data/test_briefs.json', 'r', encoding='utf-8') as file:
    test_briefs = json.load(file)

# اختيار أول حالة فقط (BR001 - مقهى للطلاب)
sample_brief = test_briefs[0]

# عرض البيانات للتأكد
print("✅ تم تحميل البيانات بنجاح!")
print(f"\n📝 حالة الاختبار المختارة:")
print(f"Industry: {sample_brief['industry']}")
print(f"Target Audience: {sample_brief['target_audience']}")
print(f"Purpose: {sample_brief['brand_purpose']}")

✅ تم تحميل البيانات بنجاح!

📝 حالة الاختبار المختارة:
Industry: coffee
Target Audience: university students
Purpose: provide a comfortable and energetic place for studying and socializing


In [19]:
system_prompt = """You are an expert Brand Strategist. 
Your task is to analyze a raw brand brief and extract a structured Brand Specification.
You MUST output ONLY valid JSON. Do not include any markdown formatting (like ```json) or extra text."""


user_prompt = f"""Analyze this brief:
Industry: {sample_brief['industry']}
Target Audience: {sample_brief['target_audience']}
Description: {sample_brief['brand_purpose']}

Return a JSON object with EXACTLY this structure:
{{
  "industry": "string",
  "target_audience": "string",
  "brand_purpose": "string",
  "personality": ["string", "string"],
  "tone": "string",
  "visual_style": ["string", "string"],
  "keywords": ["string", "string"],
  "color_direction": ["string", "string"],
  "logo_direction": "string"
}}"""

print("done")

done


In [20]:
#GPT-OSS-20B
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # اسم النموذج الدقيق من القائمة
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)
raw_output = response.choices[0].message.content
clean_json_string = raw_output.replace("```json", "").replace("```", "").strip()
brand_spec = json.loads(clean_json_string)
result_1 = json.loads(clean_json)
print("✅ نجح GPT-OSS-20B! إليك النتيجة:")
print(json.dumps(brand_spec, indent=2, ensure_ascii=False))

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Key limit exceeded (total limit). Manage it using https://openrouter.ai/workspaces/default/keys/1d42c1a2288188a34728f23eeac5b5964959ca1227425651001afdbb339c6541', 'code': 403}}

In [ ]:
#gpt-oss-120b

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_2 = json.loads(clean_json)

print("✅ النتيجة من openai/gpt-oss-120b:")
print(json.dumps(result_2, indent=2, ensure_ascii=False))

In [ ]:
#qwen3.8-27b

response = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
    response_format={"type": "json_object"}
)

raw_output = response.choices[0].message.content
clean_json = raw_output.replace("```json", "").replace("```", "").strip()
result_3 = json.loads(clean_json)

print("✅ النتيجة من qwen/qwen3.8-27b:")
print(json.dumps(result_3, indent=2, ensure_ascii=False))

#  Comparing 3 models 
---
   - qwen/qwen3.8-27b
   -  openai/gpt-oss-20b
   -  openai/gpt-oss-120b

In [ ]:
import pandas as pd

comparison_data = [
    {
        "Model": "GPT-OSS-20B",
        "Personality": ", ".join(result_1.get('personality', [])),
        "Tone": result_1.get('tone', ''),
        "Visual Style": ", ".join(result_1.get('visual_style', [])),
        "Color Direction": ", ".join(result_1.get('color_direction', [])),
        "Keywords": ", ".join(result_1.get('keywords', [])),
        "Logo Direction": result_1.get('logo_direction', '')
    },
    {
        "Model": "GPT-OSS-120B",
        "Personality": ", ".join(result_2.get('personality', [])),
        "Tone": result_2.get('tone', ''),
        "Visual Style": ", ".join(result_2.get('visual_style', [])),
        "Color Direction": ", ".join(result_2.get('color_direction', [])),
        "Keywords": ", ".join(result_2.get('keywords', [])),
        "Logo Direction": result_2.get('logo_direction', '')
    },
    {
        "Model": "Qwen 3.8 27B",
        "Personality": ", ".join(result_3.get('personality', [])),
        "Tone": result_3.get('tone', ''),
        "Visual Style": ", ".join(result_3.get('visual_style', [])),
        "Color Direction": ", ".join(result_3.get('color_direction', [])),
        "Keywords": ", ".join(result_3.get('keywords', [])),
        "Logo Direction": result_3.get('logo_direction', '')
    }
]


df_full_comparison = pd.DataFrame(comparison_data)



display(df_full_comparison)